# Segmentation

Segmentation reduces the temporal resolution within each cluster representative by
grouping consecutive timesteps into segments. This is useful when
optimizing over cluster representatives — fewer timesteps means faster solves.

tsam_xarray exposes `segment_durations` as a DataArray and provides
`disaggregate()` to map segmented results back to the original time axis.

In [ ]:
import plotly.io as pio
import xarray as xr
import xarray_plotly  # noqa: F401
from tsam import SegmentConfig

import tsam_xarray
from tsam_xarray._sample_data import sample_energy_data

pio.renderers.default = "notebook_connected"

da = sample_energy_data(n_days=30).sel(region="north", scenario="low")
print(f"Input shape: {dict(da.sizes)}")

## Aggregate with segmentation

Pass `segments=SegmentConfig(n_segments=6)` to reduce each 24-hour period to 6 segments.

In [ ]:
result = tsam_xarray.aggregate(
    da,
    time_dim="time",
    cluster_dim="variable",
    n_clusters=4,
    segments=SegmentConfig(n_segments=6),
)
n_compact = result.n_clusters * result.n_timesteps_per_period
print(f"Cluster representatives: {dict(result.cluster_representatives.sizes)}")
print(f"Data reduction: {da.sizes['time']} -> {n_compact} values")
result.cluster_representatives.to_dataframe("value").head(10)

In [ ]:
result.cluster_representatives.plotly.line(
    x="timestep",
    facet_col="variable",
    color="cluster",
    title="Segmented cluster representatives (6 segments per day)",
)

## Segment durations

Each segment spans a different number of original timesteps.
`segment_durations` tells you how many hours each segment represents.

In [ ]:
result.segment_durations.to_dataframe("hours")

Durations sum to 24 (hours per day) for each cluster:

In [ ]:
result.segment_durations.sum(dim="timestep").to_dataframe("total_hours")

## Accuracy

In [ ]:
result.accuracy.rmse.to_dataframe("RMSE")

## Disaggregate

With segmentation, `disaggregate()` places values at segment boundaries
and fills the rest with NaN. You choose how to fill.

In [ ]:
dis = result.disaggregate(result.cluster_representatives)
print(f"NaN values: {int(dis.isnull().sum())} / {dis.size}")
dis.sel(variable="solar").to_dataframe("value").head(30)

### Forward-fill for rate variables (power, temperature)

In [ ]:
filled = dis.ffill(dim="time")

comparison = xr.concat(
    [da.sel(variable="solar"), filled.sel(variable="solar")],
    dim="source",
).assign_coords(source=["original", "disaggregated + ffill"])
comparison.plotly.line(
    x="time",
    color="source",
    title="Segmented disaggregation vs original (solar)",
)

### Comparison: with and without segmentation

In [ ]:
result_noseg = tsam_xarray.aggregate(
    da,
    time_dim="time",
    cluster_dim="variable",
    n_clusters=4,
)

comparison = xr.concat(
    [
        da.sel(variable="solar"),
        result_noseg.reconstructed.sel(variable="solar"),
        filled.sel(variable="solar"),
    ],
    dim="source",
).assign_coords(source=["original", "4 clusters (no seg)", "4 clusters x 6 segments"])
comparison.plotly.line(
    x="time",
    color="source",
    title="Effect of segmentation on reconstruction quality (solar)",
)